# L5–L6 Kernel Method — Complete Integrated Lecture Notes

These notes follow the pages that are actually visible in the two slide decks and connect L5 and L6 into a self-contained learning resource. The two lectures are not isolated topics; together they form one continuous line of reasoning:

$$
\text{instability and overfitting in ordinary least squares}
\rightarrow
\text{regularization and bias--variance}
\rightarrow
\text{nonlinear features}
\rightarrow
\text{the kernel trick}
\rightarrow
\text{mathematical conditions for valid kernels}
\rightarrow
\text{kernel SVM, kernel ridge regression, and RKHSs}
\rightarrow
\text{kernel regression and attention}.
$$

### Overall structure

| Lecture and pages | Topic block | Central question |
|---|---|---|
| L5 pp. 1–12 | Ridge regression and regularization | Why can deliberately introducing bias reduce test error? |
| L5 pp. 13–20 | Nonlinear classification and feature maps | How can a nonlinear problem in the input space become a linear problem in a feature space? |
| L5 pp. 21–25 | Where features come from | Should features be engineered explicitly, used implicitly, or learned from data? |
| L5 pp. 26–30 | Nonlinear SVMs and the kernel trick | Why need an SVM never construct its high-dimensional features explicitly? |
| L6 pp. 2–13 | Kernel theory | Which functions are genuine kernels, and how can they be constructed and verified? |
| L6 pp. 14–20 | Kernel ridge regression, RKHSs, and the representer theorem | Why does a kernel model ultimately require only finitely many coefficients, one per training example? |
| L6 pp. 22–24 | KDE, kernel regression, and attention | In what sense can attention be viewed as regression with learned similarities? |

The book-based additions integrated below come from:

- [ltfp_book.pdf](</E:/Master Lecture/Mathmatical MechineLearning/slides/ltfp_book.pdf>), which supplies the main theoretical development. Its PDF page number is 16 larger than its printed page number.
- [MLbookSol.pdf](</E:/Master Lecture/Mathmatical MechineLearning/slides/MLbookSol.pdf>), a solution manual that supplies examples, counterexamples, and additional derivations. A few typographical errors in it are identified in the relevant sections below.

## Section 1 — Problems with Ordinary Least Squares (L5 p. 3)

### L5: Completing Regularization and Beginning Kernels I

Let the training data be

$$
S=\{(x_1,y_1),\ldots,(x_N,y_N)\},
\qquad x_i\in\mathbb R^d,\quad y_i\in\mathbb R.
$$

Place the examples in the rows of the design matrix:

$$
X=
\begin{bmatrix}
x_1^\top\\
\vdots\\
x_N^\top
\end{bmatrix}
\in\mathbb R^{N\times d},
\qquad
y\in\mathbb R^N,\quad w\in\mathbb R^d.
$$

Ordinary least squares solves

$$
\min_w L(w)
=
\sum_{i=1}^N(y_i-w^\top x_i)^2
=
\|Xw-y\|^2.
$$

Differentiating gives

$$
\nabla L(w)=2X^\top(Xw-y).
$$

Setting the gradient equal to zero gives the normal equations

$$
X^\top Xw=X^\top y.
$$

If $X^\top X$ is invertible, then

$$
\boxed{
\hat w_{\mathrm{LS}}
=(X^\top X)^{-1}X^\top y.
}
$$

### What do $d<N$ and $d>N$ mean?

- When $d<N$, there are more examples than features, but this alone does not guarantee that $X^\top X$ is invertible. We additionally need

  $$
  \operatorname{rank}(X)=d,
  $$

  so that $X$ has full column rank.

- When $d>N$,

  $$
  \operatorname{rank}(X)\le N<d,
  $$

  so $X^\top X$ must be singular. The least-squares solution is generally nonunique.

- Even when $d<N$, highly collinear features can make $X^\top X$ nearly singular. Its inverse may exist in theory while the numerical result remains extremely sensitive to tiny perturbations of the data.

This leads to the first question of the lecture: how can we make the problem both solvable and stable?

### Supplement: “linear” means linear in the parameters

Source: *ltfp*, Sections 3.2–3.5, PDF pp. 62–70.

A general linear model has the form

$$
f_\theta(x)=\phi(x)^\top\theta.
$$

The model is linear in the parameter $\theta$, but it can be highly nonlinear in the original input $x$. For example,

$$
\phi(x)=(1,x,x^2,\sin x,\ldots).
$$

Thus, “linear regression” does not mean “a model that can fit only straight lines.” Its expressive power is determined by the feature map $\phi$.

### Supplement: OLS is an orthogonal projection

Let $\Phi\in\mathbb R^{n\times d}$ be a full-column-rank design matrix. Then

$$
\hat\theta_{\rm OLS}
=(\Phi^\top\Phi)^{-1}\Phi^\top y.
$$

The fitted training vector is

$$
\hat y=\Phi\hat\theta_{\rm OLS}=P_\Phi y,
$$

where

$$
P_\Phi=\Phi(\Phi^\top\Phi)^{-1}\Phi^\top
$$

is the orthogonal projection onto $\operatorname{im}(\Phi)$. Consequently, the residual is orthogonal to every column of $\Phi$:

$$
\Phi^\top(y-\hat y)=0.
$$

### Supplement: do not explicitly form an inverse in numerical work

The closed-form inverse is mainly useful for theoretical analysis. Explicitly constructing

$$
(\Phi^\top\Phi)^{-1}
$$

in numerical computation magnifies conditioning problems and loses precision. In practice, prefer:

- QR decomposition;
- Cholesky decomposition;
- conjugate gradients;
- gradient descent or stochastic gradient descent.

### Supplement: fixed and random design must be distinguished

A fixed-design analysis treats $x_1,\ldots,x_n$ as deterministic and repeatedly generates only the noise and labels. It measures denoising performance at those already observed inputs.

A random-design analysis instead assumes

$$
X\sim p_X
$$

and measures generalization to new inputs. A fixed-design bias–variance formula cannot automatically be interpreted as a random-design generalization result.

Under a correctly specified linear model with a full-rank fixed design,

$$
\mathbb E[R(\hat\theta_{\rm OLS})]-R^*
=\frac{\sigma^2d}{n}.
$$

If test labels are regenerated at the same input locations, then

$$
\mathbb E[\operatorname{MSE}_{\rm train}]
=\sigma^2\left(1-\frac dn\right),
$$

whereas

$$
\mathbb E[\operatorname{MSE}_{\rm test}]
=\sigma^2\left(1+\frac dn\right).
$$

On average, training error therefore underestimates test error by

$$
\frac{2\sigma^2d}{n}.
$$

This quantifies why adding parameters can lower training error while worsening generalization.

## Section 2 — From Adding $\lambda I$ to Ridge Regression (L5 pp. 4–5)

The slides first propose what looks like a purely algebraic repair:

$$
X^\top X
\quad\longmapsto\quad
X^\top X+\lambda I,
\qquad \lambda>0.
$$

The resulting estimator is

$$
\boxed{
\hat w_\lambda
=(X^\top X+\lambda I)^{-1}X^\top y.
}
$$

### Why is the matrix always invertible?

Because

$$
v^\top X^\top Xv=\|Xv\|^2\ge0,
$$

$X^\top X$ is positive semidefinite. For every nonzero $v$,

$$
v^\top(X^\top X+\lambda I)v
=\|Xv\|^2+\lambda\|v\|^2>0.
$$

Therefore $X^\top X+\lambda I$ is positive definite and hence invertible.

From the eigenvalue perspective, if the eigenvalues of $X^\top X$ are

$$
\mu_1,\ldots,\mu_d,
\qquad \mu_j\ge0,
$$

then regularization changes them to

$$
\mu_j+\lambda>0.
$$

It removes zero eigenvalues and raises very small eigenvalues, thereby improving the condition number.

The slides note that making $X^\top X$ nonsingular was the original motivation for ridge regression when Hoerl and Kennard introduced it in 1970.

### Which optimization problem corresponds to this algebraic modification?

Ridge regression can be written as

$$
\min_w
\frac12\|Xw-y\|^2+\lambda\|w\|^2.
$$

It asks for both:

1. a small training error;
2. a parameter vector whose norm is not too large.

Thus ridge is not merely a matrix-inversion trick. It deliberately introduces a preference into estimation:

> Among models with comparable fit, prefer parameters that are smaller and more stable.

Large parameter values often indicate:

- extreme sensitivity to small changes in the inputs or training examples;
- huge mutually cancelling coefficients along nearly collinear directions;
- violent fluctuations introduced to fit noise.

### The factor of two in the slides

The slides write

$$
\frac12\|Xw-y\|^2+\lambda\|w\|^2
$$

but display a solution involving $X^\top X+\lambda I$. If exactly the same $\lambda$ is used when differentiating, the result is $X^\top X+2\lambda I$.

This does not change the conclusion: the factor of two is normally absorbed into the definition of the hyperparameter $\lambda$. Different texts may show

$$
X^\top X+\lambda I,
\qquad
X^\top X+2\lambda I,
\qquad
X^\top X+N\lambda I,
$$

depending on whether the loss is divided by $N$ and whether a factor $1/2$ is placed before the regularizer.

### Supplement: primal and dual ridge formulas

Source: *ltfp*, Sections 3.6–3.9, PDF pp. 72–84.

For the objective

$$
\min_\theta
\frac1n\|y-\Phi\theta\|_2^2+\lambda\|\theta\|_2^2,
$$

the solution is

$$
\boxed{
\hat\theta_\lambda
=(\Phi^\top\Phi+n\lambda I_d)^{-1}\Phi^\top y
}
$$

and equivalently

$$
\boxed{
\hat\theta_\lambda
=\Phi^\top(\Phi\Phi^\top+n\lambda I_n)^{-1}y.
}
$$

Consequently:

- when $d\ll n$, solve the $d\times d$ primal system;
- when $d\gg n$, solve the $n\times n$ dual system;
- in an infinite-dimensional feature space, the dual formula remains usable as long as $\Phi\Phi^\top$ can be evaluated.

This identity is precisely the bridge from ridge regression to kernel ridge regression.

## Section 3 — Why Regularization Can Improve Test Error: Bias–Variance Decomposition (L5 pp. 6–8)

Suppose the true observation model is

$$
y=f(x)+\eta,
$$

where $f(x)$ is the true noise-free regression function and $\eta$ is zero-mean noise.

Training produces a random predictor $\hat f$. It is random because the training set, observation noise, and even the optimization procedure may vary.

The quantity of interest is not training error but

$$
\mathbb E\|y-\hat f\|^2.
$$

First add and subtract $f$:

$$
y-\hat f=(y-f)+(f-\hat f).
$$

Expanding the square yields

$$
\|y-\hat f\|^2
=
\|y-f\|^2+\|f-\hat f\|^2
+2\langle y-f,f-\hat f\rangle.
$$

Since $y-f=\eta$, the expected cross term vanishes under the zero-mean and appropriate independence or uncorrelatedness assumptions:

$$
\mathbb E\|y-\hat f\|^2
=
\mathbb E\|\eta\|^2
+
\mathbb E\|f-\hat f\|^2.
$$

Now add and subtract the mean predictor $\mathbb E\hat f$ in the second term:

$$
f-\hat f
=
f-\mathbb E\hat f+\mathbb E\hat f-\hat f.
$$

Expanding once more and taking expectations gives

$$
\mathbb E\|f-\hat f\|^2
=
\|f-\mathbb E\hat f\|^2
+
\mathbb E\|\hat f-\mathbb E\hat f\|^2.
$$

Therefore,

$$
\boxed{
\mathbb E\|y-\hat f\|^2
=
\underbrace{\mathbb E\|\eta\|^2}_{\text{irreducible noise}}
+
\underbrace{\|f-\mathbb E\hat f\|^2}_{\text{squared bias}}
+
\underbrace{\mathbb E\|\hat f-\mathbb E\hat f\|^2}_{\text{variance}}.
}
$$

The three terms mean:

- **irreducible noise:** randomness inherent in the data-generating mechanism;
- **squared bias:** how far the average model over repeated training sets lies from the true function;
- **variance:** how much the fitted model changes when the training set changes.

### Why can a biased model be better?

Under the classical assumptions, ordinary least squares is BLUE:

> Best Linear Unbiased Estimator.

This is the Gauss–Markov theorem. However, “best” means only that OLS has the smallest variance among estimators that are both linear and unbiased. It does not imply that:

- OLS is better than every biased estimator;
- OLS necessarily has the smallest test mean-squared error;
- OLS is better than every nonlinear estimator.

Ridge regression deliberately introduces bias but can greatly reduce variance. Whenever the variance reduction exceeds the increase in squared bias, test error decreases.

### Supplement: two different error decompositions

Source: *ltfp*, Sections 3.6–3.9 and 4.5.4–4.5.6; *MLbookSol*, pp. 34–38 and 65.

Do not treat the following two decompositions as the same formula.

For squared loss, the bias–variance decomposition is

$$
\text{total error}
=
\text{irreducible noise}
+\text{squared bias}
+\text{variance}.
$$

Learning theory instead uses an approximation–estimation decomposition:

$$
L_D(\hat h)-L_D^*
=
\underbrace{L_D(h_{\mathcal H}^*)-L_D^*}_{\text{approximation error}}
+
\underbrace{L_D(\hat h)-L_D(h_{\mathcal H}^*)}_{\text{estimation error}}.
$$

Enlarging the model class generally lowers approximation error but raises estimation error.

## Section 4 — Ridge Regression Is Indeed Biased (L5 p. 9)

Use the averaged-loss convention:

$$
\min_w
\frac1N\|Xw-y\|^2+\lambda\|w\|^2.
$$

The solution is

$$
\hat w_{\mathrm{RR}}
=(X^\top X+N\lambda I)^{-1}X^\top y.
$$

Let

$$
M=X^\top X.
$$

Ordinary least squares is

$$
\hat w_{\mathrm{LS}}=M^{-1}X^\top y.
$$

Hence

$$
\begin{aligned}
\hat w_{\mathrm{RR}}
&=(M+N\lambda I)^{-1}X^\top y\\
&=(M+N\lambda I)^{-1}M\hat w_{\mathrm{LS}}\\
&=(I+N\lambda M^{-1})^{-1}\hat w_{\mathrm{LS}}.
\end{aligned}
$$

Define

$$
A=(I+N\lambda M^{-1})^{-1}.
$$

Then

$$
\hat w_{\mathrm{RR}}=A\hat w_{\mathrm{LS}}.
$$

Assume

$$
y=Xw_{\mathrm{true}}+\eta,
\qquad
\mathbb E\eta=0.
$$

Because OLS is unbiased,

$$
\mathbb E\hat w_{\mathrm{LS}}=w_{\mathrm{true}}.
$$

Therefore,

$$
\boxed{
\mathbb E\hat w_{\mathrm{RR}}
=Aw_{\mathrm{true}}
\ne w_{\mathrm{true}}
}
$$

in general whenever $\lambda>0$.

The bias vector is

$$
\operatorname{Bias}(\hat w_{\mathrm{RR}})
=(A-I)w_{\mathrm{true}}
=-N\lambda(M+N\lambda I)^{-1}w_{\mathrm{true}}.
$$

### Variance exercise left by the slides

If

$$
\operatorname{Var}(\eta)=\sigma^2I,
$$

then

$$
\operatorname{Var}(\hat w_{\mathrm{LS}})
=\sigma^2M^{-1}.
$$

Consequently,

$$
\boxed{
\operatorname{Var}(\hat w_{\mathrm{RR}})
=A\operatorname{Var}(\hat w_{\mathrm{LS}})A^\top
=\sigma^2(M+N\lambda I)^{-1}
M
(M+N\lambda I)^{-1}.
}
$$

For an eigenvector of $M$ with eigenvalue $\mu_j$:

- the OLS variance is

  $$
  \frac{\sigma^2}{\mu_j};
  $$

- the ridge variance is

  $$
  \frac{\sigma^2\mu_j}{(\mu_j+N\lambda)^2};
  $$

- the mean shrinkage factor is

  $$
  \frac{\mu_j}{\mu_j+N\lambda}.
  $$

When $\mu_j$ is small, the OLS variance can be enormous. Ridge regression shrinks precisely these weakly informed, ill-conditioned directions most strongly.

### Supplement: exact spectral bias–variance formula for ridge

Let

$$
\widehat\Sigma=\frac1n\Phi^\top\Phi
=U\operatorname{diag}(\mu_1,\ldots,\mu_d)U^\top,
\qquad
a_j=u_j^\top\theta_*.
$$

Under fixed design,

$$
\begin{aligned}
\mathbb E[R(\hat\theta_\lambda)]-R^*
={}&
\lambda^2\theta_*^\top
(\widehat\Sigma+\lambda I)^{-2}
\widehat\Sigma\theta_*\\
&+\frac{\sigma^2}{n}
\operatorname{tr}\!\left[
\widehat\Sigma^2(\widehat\Sigma+\lambda I)^{-2}
\right].
\end{aligned}
$$

Direction by direction,

$$
\operatorname{Bias}^2
=
\sum_j
\frac{\lambda^2\mu_j}{(\mu_j+\lambda)^2}a_j^2,
$$

and

$$
\operatorname{Variance}
=
\frac{\sigma^2}{n}
\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

Thus ridge does not simply multiply every coefficient by the same constant:

- if $\mu_j\gg\lambda$, the data strongly support that direction, so much of it is retained;
- if $\mu_j\ll\lambda$, the data barely support that direction, so it is strongly suppressed.

## Section 5 — $\lambda$ Controls Model Complexity (L5 p. 10)

The general tendency is:

- a more complex model has lower bias and higher variance;
- a simpler model has higher bias and lower variance.

In ridge regression, $\lambda$ is the control knob:

- $\lambda=0$: ordinary least squares and the greatest freedom;
- small $\lambda$: weak shrinkage, lower bias, and higher variance;
- large $\lambda$: parameters are pulled toward zero, giving higher bias and lower variance.

The figure in the slides illustrates that:

- squared bias increases with $\lambda$;
- variance decreases with $\lambda$;
- their sum usually has a minimum;
- the minimum of test error is close to that minimum.

Therefore, $\lambda$ should generally not be selected by training error alone. A validation set, cross-validation, or a related model-selection method is required.

### Supplement: two notions of effective degrees of freedom

The quantity controlling fixed-design variance in *ltfp* is

$$
df_2(\lambda)
=
\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

Statistical literature also commonly calls the trace of the smoothing matrix

$$
df_1(\lambda)
=
\sum_j
\frac{\mu_j}{\mu_j+\lambda}
$$

the effective degrees of freedom. The two quantities should not be confused:

- $df_1=\operatorname{tr}(H_\lambda)$ is the model's soft parameter count;
- $df_2=\operatorname{tr}(H_\lambda^2)$ appears directly in predictive variance.

### Supplement: a risk bound with no explicit dimension dependence

Let

$$
\operatorname{tr}(\widehat\Sigma)
=\frac1n\sum_i\|\phi(x_i)\|^2.
$$

Balancing upper bounds on bias and variance gives

$$
\lambda_*
=
\frac{\sigma\sqrt{\operatorname{tr}(\widehat\Sigma)}}
{\|\theta_*\|\sqrt n},
$$

and

$$
\mathbb E[R(\hat\theta_{\lambda_*})]-R^*
\le
\frac{
\sigma\|\theta_*\|
\sqrt{\operatorname{tr}(\widehat\Sigma)}
}{\sqrt n}.
$$

This bound has no explicit dependence on $d$. It shows that dimension alone is not always the best measure of complexity; the parameter norm and feature norms can matter more.

However, $\lambda_*$ depends on the unknown $\sigma$ and $\theta_*$, so cross-validation is still needed in practice.

## Section 6 — The Full Motivation for Regularization (L5 pp. 11–12)

Regularization is not only a device for preventing a singular matrix. The slides give four more general motivations.

### 6.1 Underdetermined problems

If $d>N$, then $X^\top X$ must be singular, whereas ridge regression still gives a unique solution.

### 6.2 Ill-conditioning and numerical stability

Even when $N\ge d$, the matrix can be nearly singular, causing:

- difficulty in obtaining an accurate solution;
- amplification of floating-point error;
- huge parameter changes in response to tiny data changes;
- poor stability and robustness.

### 6.3 Preventing the fitting of noise

High-order features or complex models can drive training error very low while following every noisy observation, thereby increasing error on unseen data.

### 6.4 Injecting prior structure

If we know what a reasonable model should look like, that structure can be inserted through:

- a Bayesian prior;
- parameter constraints;
- penalty terms.

The general regularized objective is

$$
\boxed{
\min_w
\frac1N\sum_i(y_i-w^\top x_i)^2
+
\lambda\Omega(w).
}
$$

Here:

- empirical risk is responsible for fitting the data;
- $\Omega(w)$ determines which models are preferred;
- $\lambda$ controls the relative strength of data fit and structural preference.

For example:

- $\Omega(w)=\|w\|_2^2$ favors small, smoothly varying parameters;
- $\Omega(w)=\|w\|_1$ usually promotes sparsity;
- other regularizers can express low-rank, grouped, or smooth structure.

The slides also emphasize that regularization need not appear explicitly as $\lambda\Omega(w)$. Model architecture, the optimization algorithm, computational constraints, and data augmentation can all induce implicit regularization.

### Supplement: Bayesian interpretation of ridge

If

$$
\theta_*
\sim
\mathcal N\!\left(
0,\frac{\sigma^2}{n\lambda}I
\right),
\qquad
y\mid\theta_*
\sim\mathcal N(\Phi\theta_*,\sigma^2I),
$$

then ridge is simultaneously:

- the posterior mean;
- the posterior mode, or MAP estimator.

L2 regularization can therefore be interpreted as a Gaussian prior expressing the belief that the parameters lie near zero and have no exceptionally large direction.

### Supplement: regularization and stability

*MLbookSol* gives a general result. If an algorithm has average stability $\epsilon_{\rm stab}$ and is also an $\epsilon_{\rm ERM}$-approximate empirical risk minimizer, then

$$
\boxed{
\mathbb E[
L_D(A(S))-L_D(h^*)
]
\le
\epsilon_{\rm stab}+\epsilon_{\rm ERM}.
}
$$

For an L2-regularized problem with a Lipschitz loss, the stability term typically satisfies

$$
\epsilon_{\rm stab}
=
O\!\left(
\frac{G^2R^2}{\lambda n}
\right).
$$

Therefore:

- increasing $\lambda$ makes the algorithm more stable and lowers estimation error;
- but it increases regularization bias.

This is another explanation of why regularization can improve generalization, beyond the statement that it reduces parameter variance.

### Supplement: the fundamental difference between L1 and L2

For the one-dimensional problem

$$
\min_w
\frac12w^2-xw+\lambda|w|,
$$

the solution is the soft-thresholding rule

$$
\boxed{
w=\operatorname{sign}(x)(|x|-\lambda)_+.
}
$$

Thus L1 can shrink small coefficients exactly to zero, whereas L2 generally shrinks them continuously without producing exact zeros. This explains the feature-selection ability of the Lasso.

## Section 7 — Two Routes to Nonlinear Classification (L5 pp. 13–17)

### L5 Part II: From Nonlinear Classification to Feature Maps

There are two broad strategies:

1. use a nonlinear classifier directly on the original inputs;
2. first apply a nonlinear feature map

   $$
   x\mapsto\phi(x),
   $$

   and then use a linear classifier:

   $$
   h(x)=\operatorname{sgn}\bigl(w^\top\phi(x)+w_0\bigr).
   $$

The second method is linear in feature space but produces a nonlinear boundary in the original input space.

### Page 14: an elliptical classifier

Predict $+1$ when

$$
(x_1-c_1)^2+b(x_2-c_2)^2\le1
$$

and predict $-1$ otherwise.

This gives an elliptical boundary in the input space. Expanding the expression gives

$$
x_1^2+b x_2^2
-2c_1x_1-2bc_2x_2
+c_1^2+bc_2^2-1\le0.
$$

It is therefore a linear decision rule on the features

$$
\phi(x)=(1,x_1,x_2,x_1^2,x_2^2).
$$

### Page 15: multiple hyperplanes

When one line cannot enclose a central region, several half-spaces can be combined to form a polygonal or piecewise-linear decision region.

Each individual boundary is linear, but the logical conjunction “satisfy all of these linear inequalities” already defines a nonlinear classifier.

### Page 16: $k$-nearest neighbors

The slides show the irregular decision boundary of a 15-nearest-neighbor classifier.

$k$NN does not explicitly construct an analytic boundary. Instead, it:

1. finds the $k$ training points closest to a new query;
2. predicts by the majority class among those neighbors.

The boundary is determined by the local distribution of the data and can therefore be highly nonlinear. Typically:

- small $k$: a complex boundary, low bias, and high variance;
- large $k$: a smoother boundary, high bias, and low variance.

### Supplement: every finite hypothesis class can be linearized

Source: *MLbookSol*, p. 41.

Let

$$
\mathcal H=\{h_1,\ldots,h_M\}.
$$

Define

$$
\phi(x)=
(\mathbf1\{h_1(x)=1\},\ldots,
\mathbf1\{h_M(x)=1\},1).
$$

For the $j$th hypothesis, take

$$
w_j=(2e_j,-1).
$$

Then

$$
\langle w_j,\phi(x)\rangle
=2\mathbf1\{h_j(x)=1\}-1
=h_j(x).
$$

The obstacle is therefore not whether a nonlinear rule can be made linear. Rather, the explicit feature dimension may be enormous, motivating the kernel trick or approximate features.

## Section 8 — Hand-Crafted Nonlinear Features (L5 pp. 18–20)

### 8.1 A one-dimensional V-shaped map

Suppose the middle points belong to one class and the two ends belong to the other. No single threshold on the original line can assign both ends to the same class while separating the middle.

Use

$$
\boxed{
\phi(x)=(x,|x|).
}
$$

After the mapping:

- middle points have small $|x|$ and lie near the bottom of the V;
- points at either end have large $|x|$ and lie higher on the V.

A horizontal line in the resulting two-dimensional space can now separate the two classes.

### 8.2 XOR-type data and interaction features

In the original two-dimensional space, opposite diagonal quadrants have the same class, so no single line can separate the data.

Map the input to

$$
\boxed{
\phi(x_1,x_2)
=(x_1,x_1x_2,x_2).
}
$$

The key feature is $x_1x_2$:

- when $x_1$ and $x_2$ have the same sign, $x_1x_2>0$;
- when they have different signs, $x_1x_2<0$.

The sign of the new coordinate directly distinguishes the two diagonal patterns, making them linearly separable in three dimensions.

### 8.3 Why does the feature dimension explode?

For $p$ original variables:

- the linear features

  $$
  (x_1,\ldots,x_p)
  $$

  have dimension $p$;

- all distinct quadratic monomials $x_ix_j$ with $i\le j$ have dimension

  $$
  \boxed{\frac{p(p+1)}2};
  $$

- all polynomial monomials of total degree at most $d$ have:

  - dimension

    $$
    \binom{p+d}{d}
    $$

    when the constant term is included;

  - dimension

    $$
    \boxed{\binom{p+d}{d}-1}
    $$

    when the constant term is excluded.

The number of features grows combinatorially with the input dimension and polynomial degree, and some useful feature spaces are infinite-dimensional. This is why the later algorithms avoid constructing $\phi(x)$ explicitly.

### Supplement: exact polynomial feature dimensions

Source: *ltfp*, Section 7.3.1, PDF pp. 202–203; *MLbookSol*, p. 41.

The homogeneous polynomial kernel

$$
k_s(x,z)=(x^\top z)^s
$$

corresponds to all monomials of total degree exactly $s$. Its smallest explicit feature dimension is

$$
\boxed{
\binom{d+s-1}{s}.
}
$$

The inhomogeneous kernel

$$
k_s(x,z)=(1+x^\top z)^s
$$

contains every monomial of degree at most $s$, and its dimension is

$$
\boxed{
\binom{d+s}{s}.
}
$$

The quantity $d^s$ used elsewhere in the notes is the ordered-tensor dimension before repeated monomials are merged. Merging those duplicates yields the binomial-coefficient dimensions above.

## Section 9 — How Features Can Be Obtained (L5 pp. 21–25)

The slides distinguish three paradigms.

### Explicit feature extraction

$$
\text{raw data}
\rightarrow
\text{feature vector}
\rightarrow
\text{classifier}.
$$

For example, an email can be converted into a bag-of-words, term-frequency, or TF–IDF vector.

### Implicit use of nonlinear features

Use a nonlinear classifier that can be understood as a linear method in a feature space that is unknown or never constructed explicitly.

Kernel methods belong to this category.

### Learning features from data

Instead of fixing $\phi$ by hand, let training learn the representation itself. Deep neural networks are the standard example.

The route chosen in the remainder of this lecture is:

> Specify the feature map implicitly and access it only through inner products.

## Section 10 — Hard-Margin and Soft-Margin SVMs (L5 pp. 26–27)

### L5 Part III: From SVMs to the Kernel Trick

The hard-margin SVM solves

$$
\min_{w,w_0}\frac12\|w\|^2
$$

subject to

$$
y_i(w^\top x_i+w_0)\ge1.
$$

Minimizing $\|w\|$ is equivalent to maximizing the geometric margin. Hard margin requires the training data to be strictly linearly separable.

The soft-margin SVM introduces $\xi_i\ge0$:

$$
\boxed{
\min_{w,w_0,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
}
$$

subject to

$$
y_i(w^\top x_i+w_0)\ge1-\xi_i,
\qquad
\xi_i\ge0.
$$

The slack variables have the following interpretation:

- $\xi_i=0$: the point satisfies the standard margin requirement;
- $0<\xi_i\le1$: the point is classified correctly but lies inside the margin;
- $\xi_i>1$: the point is misclassified.

A larger $C$ tolerates fewer training violations. A smaller $C$ permits more violations in exchange for a wider margin and stronger regularization.

### Supplement: the relation between $C$ and $\lambda$

Source: *ltfp*, Section 4.1.2, PDF pp. 90–95; *ltfp*, Section 12.1.2, PDF pp. 362–364; *MLbookSol*, pp. 39–44.

The standard soft-margin formulation

$$
\min_{w,b,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
$$

and the regularized hinge-risk formulation

$$
\min_{w,b}
\frac1n\sum_i
(1-y_i(w^\top x_i+b))_+
+\frac\lambda2\|w\|^2
$$

are related by

$$
\boxed{\lambda=\frac1{nC}}.
$$

Thus:

- large $C$ puts more weight on training violations and gives weaker regularization;
- small $C$ permits more violations and gives stronger regularization.

### Supplement: SVM dual variables and types of support vectors

The dual is

$$
\max_\alpha
\sum_i\alpha_i
-\frac12
\sum_{i,j}
\alpha_i\alpha_jy_iy_jK(x_i,x_j),
$$

subject to

$$
0\le\alpha_i\le C,
\qquad
\sum_i\alpha_i y_i=0.
$$

The KKT conditions imply:

- $y_if(x_i)>1\Rightarrow\alpha_i=0$: the point does not affect the final prediction;
- $0<\alpha_i<C\Rightarrow y_if(x_i)=1$: the point lies exactly on a margin boundary;
- $\alpha_i=C$: the point may lie inside the margin or be misclassified.

Support-vector sparsity can be computationally useful. In high-dimensional problems, however, the number of support vectors can be proportional to $n$, so statistical performance cannot be explained solely by saying that there are few support vectors.

### Supplement: separability does not make soft margin identical to hard margin

*MLbookSol* gives a counterexample containing a separable point extremely close to the decision boundary. To classify it correctly, hard-margin SVM requires a very large $\|w\|$. A soft-margin SVM with finite $C$ may prefer to violate that one point in exchange for a much smaller parameter norm.

Therefore:

> Separability guarantees only that the hard-margin problem is feasible; a soft-margin SVM with a fixed finite $C$ need not return the hard-margin solution.

### Supplement: Lipschitz continuity and calibration of hinge loss

If $\|x\|\le R$, the hinge loss

$$
\ell(w;(x,y))=(1-yw^\top x)_+
$$

satisfies

$$
|\ell(w_1)-\ell(w_2)|
\le
R\|w_1-w_2\|.
$$

This allows parameter stability to be translated directly into loss stability.

Hinge loss is also classification calibrated:

$$
R_{01}(f)-R_{01}^*
\le
R_{\rm hinge}(f)-R_{\rm hinge}^*.
$$

Thus, optimizing hinge risk close to its optimum controls the corresponding excess 0–1 classification risk.

### Supplement: the margin also controls perceptron updates

If $\|x_i\|\le\rho$ and a unit vector separates the data with margin $\gamma$, then the number of perceptron mistakes satisfies

$$
\boxed{
T_{\rm mistake}\le
\left(\frac\rho\gamma\right)^2.
}
$$

A large margin therefore has both a generalization interpretation and an algorithmic consequence: it makes perceptron convergence faster.

### Supplement: implicit maximum-margin bias of logistic gradient descent

On linearly separable data, the infimum of unregularized logistic loss is zero, but there is no finite minimizer. Gradient descent satisfies

$$
\|\theta_t\|\to\infty,
$$

while its direction converges:

$$
\frac{\theta_t}{\|\theta_t\|}
\longrightarrow
\arg\max_{\|\eta\|\le1}
\min_i y_i x_i^\top\eta.
$$

The expression on the right is exactly the maximum-margin direction of the hard-margin SVM. Thus, the optimization algorithm itself can create implicit regularization.

## Section 11 — Placing an SVM in a Nonlinear Feature Space (L5 p. 28)

Replace each $x_i$ by $\phi(x_i)$:

$$
\min_{w,w_0,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
$$

subject to

$$
y_i\bigl(\langle w,\phi(x_i)\rangle+w_0\bigr)
\ge1-\xi_i.
$$

Its dual can be written as

$$
\max_\alpha
\sum_i\alpha_i
-
\frac12
\sum_{i,j}
\alpha_i\alpha_jy_iy_j
\langle\phi(x_i),\phi(x_j)\rangle
$$

subject to

$$
\sum_i\alpha_i y_i=0,
\qquad
0\le\alpha_i\le C.
$$

The Lagrangian stationarity condition gives

$$
\boxed{
w=\sum_i\alpha_i y_i\phi(x_i).
}
$$

Therefore,

$$
\begin{aligned}
\langle w,\phi(x)\rangle
&=
\sum_i\alpha_i y_i
\langle\phi(x_i),\phi(x)\rangle.
\end{aligned}
$$

The key discovery is:

> An SVM need not know the coordinates of $\phi(x)$; it needs only inner products in feature space.

## Section 12 — The Kernel Trick (L5 pp. 29–30)

If a function satisfies

$$
\boxed{
k(x,x')=\langle\phi(x),\phi(x')\rangle,
}
$$

then every feature-space inner product can be replaced directly by a kernel value:

$$
\langle w,\phi(x)\rangle
=
\sum_i\alpha_i y_i k(x_i,x).
$$

The final kernel-SVM classifier is

$$
\boxed{
h(x)
=
\operatorname{sgn}\left(
\sum_i\alpha_i y_i k(x_i,x)+w_0
\right).
}
$$

Only points with $\alpha_i>0$ appear in the final decision function. These are the support vectors.

In practice, there is no need to:

- construct the high-dimensional $\phi(x)$ explicitly;
- store the feature-space vector $w$ explicitly.

One needs only:

- the training examples;
- the coefficients $\alpha_i$;
- the labels $y_i$;
- the intercept $w_0$;
- the kernel function.

The slides' statement that one can “compute $w$” should be understood as obtaining its implicit expansion

$$
w=\sum_i\alpha_i y_i\phi(x_i),
$$

not necessarily as forming its coordinates.

This idea is not restricted to SVMs. Whenever a method's solution has the form

$$
w=\sum_i a_i\phi(x_i),
$$

the method may be kernelizable. The perceptron is another example, because each update adds a training-example feature vector to $w$.

### Supplement: the kernel perceptron

Store coefficients instead of an explicit weight vector:

$$
w^{(t)}
=
\sum_i\alpha_i^{(t)}\psi(x_i).
$$

If example $i$ is misclassified, then

$$
y_i\sum_j\alpha_jK(x_j,x_i)\le0,
$$

and the update is

$$
\alpha_i\leftarrow\alpha_i+y_i.
$$

Prediction is

$$
h(x)
=
\operatorname{sign}\left(
\sum_i\alpha_iK(x_i,x)
\right).
$$

The labels have already been absorbed into the signed coefficients $\alpha_i$; do not multiply by $y_i$ a second time.

## Section 13 — Review of the Kernel Trick and References (L6 pp. 2–3)

### L6: Kernels II

L6 begins by answering the central question left open by L5:

> An arbitrary “similarity” cannot simply be declared a kernel. Which functions really are inner products in some feature space?

The basic purpose of a kernel is to satisfy

$$
k(x,z)=\langle\phi(x),\phi(z)\rangle.
$$

If evaluating $k(x,z)$ is cheaper than explicitly constructing $\phi(x)$, learning in a high- or infinite-dimensional space becomes computationally possible.

The slides recommend:

- Schölkopf and Smola, *Learning with Kernels*;
- Berg, Christensen, and Ressel, *Harmonic Analysis on Semigroups*.

The first develops kernel methods from a machine-learning perspective; the second studies positive-definite functions and harmonic analysis in much greater depth.

## Section 14 — An Explicit Polynomial-Kernel Example (L6 p. 4)

Define

$$
\phi:\mathbb R^2\to\mathbb R^3,
\qquad
\phi(x)=
(x_1^2,\sqrt2x_1x_2,x_2^2).
$$

Then

$$
\begin{aligned}
\langle\phi(x),\phi(z)\rangle
&=
x_1^2z_1^2
+2x_1x_2z_1z_2
+x_2^2z_2^2\\
&=(x_1z_1+x_2z_2)^2\\
&=\langle x,z\rangle^2.
\end{aligned}
$$

Thus

$$
k(x,z)=\langle x,z\rangle^2
$$

implicitly represents a quadratic feature space.

More generally, on $\mathbb R^d$ define

$$
k_m(x,z)=\langle x,z\rangle^m.
$$

The slides use features indexed by ordered tuples:

$$
\phi_m(x)
=
\bigl(
x_{i_1}\cdots x_{i_m}
\bigr)_{1\le i_1,\ldots,i_m\le d}.
$$

This representation has $d^m$ coordinates, and

$$
\begin{aligned}
\langle\phi_m(x),\phi_m(z)\rangle
&=
\sum_{i_1,\ldots,i_m}
\prod_{r=1}^m x_{i_r}z_{i_r}\\
&=
\left(\sum_{j=1}^d x_jz_j\right)^m\\
&=\langle x,z\rangle^m.
\end{aligned}
$$

Explicit features require manipulating $d^m$ coordinates, whereas the kernel evaluates the original inner product in $O(d)$ time and then raises it to the $m$th power.

### Why do the slides say this $\phi$ differs from the preceding one?

When $d=2$ and $m=2$, the ordered construction is

$$
(x_1^2,x_1x_2,x_2x_1,x_2^2),
$$

whereas the compressed construction is

$$
(x_1^2,\sqrt2x_1x_2,x_2^2).
$$

The feature maps are different but generate the same kernel. Therefore,

$$
\boxed{\text{a feature map is not unique.}}
$$

After repeated monomials are merged, the standard number of features of degree exactly $m$ is

$$
\binom{d+m-1}{m}.
$$

## Section 15 — What Must a Good Kernel Satisfy? (L6 p. 5)

### Computability

$k(x,z)$ should be efficient to evaluate and, ideally, cheaper than explicitly computing an inner product between high-dimensional feature vectors.

### Relevance to the task

A mathematically valid kernel need not be appropriate for a particular problem. A kernel determines:

- which inputs count as similar;
- which functions the model prefers;
- which transformations are important or unimportant.

### Symmetry

Because inner products are symmetric,

$$
k(x,z)=k(z,x).
$$

### Positive semidefiniteness

The essential condition is that for every finite collection $x_1,\ldots,x_m$ and every set of coefficients $c_1,\ldots,c_m$,

$$
\boxed{
\sum_{i,j=1}^m
c_ic_jk(x_i,x_j)\ge0.
}
$$

If the Gram matrix is defined by

$$
K_{ij}=k(x_i,x_j),
$$

the condition is equivalent to

$$
c^\top Kc\ge0
\quad\text{for every }c.
$$

The slides often call such functions *positive definite* kernels or *Mercer kernels*. Because zero eigenvalues are allowed, *positive semidefinite kernel* is usually the more precise term.

If a feature map exists, then

$$
c^\top Kc
=
\sum_{i,j}c_ic_j\langle\phi(x_i),\phi(x_j)\rangle
=
\left\|\sum_i c_i\phi(x_i)\right\|^2
\ge0.
$$

Conversely, if every finite Gram matrix is positive semidefinite, one can construct a corresponding Hilbert feature space.

### Supplement: two especially transparent valid kernels

Source: *ltfp*, Sections 7.2–7.3, PDF pp. 197–212; *MLbookSol*, pp. 42–44.

On $\{1,\ldots,N\}$, define

$$
K(i,j)=\min(i,j).
$$

Take

$$
\psi(j)=
(\underbrace{1,\ldots,1}_{j},0,\ldots,0).
$$

Then

$$
K(i,j)=\langle\psi(i),\psi(j)\rangle.
$$

For sets $E,E'\subseteq[d]$, define

$$
\boxed{
K(E,E')=|E\cap E'|.
}
$$

This is valid because the set indicator vectors satisfy

$$
|E\cap E'|
=
\langle\mathbf1_E,\mathbf1_{E'}\rangle.
$$

These examples reinforce that inputs need not be Euclidean vectors: kernels can be defined directly on strings, sets, text, trees, graphs, and other structured objects.

## Section 16 — Algebra of Kernel Functions (L6 pp. 6–9)

If $k_1$ and $k_2$ are valid kernels, the following constructions are also valid kernels.

### Nonnegative scaling

$$
\tilde k(x,z)=c\,k_1(x,z),
\qquad c\ge0.
$$

The corresponding feature map is

$$
\tilde\phi(x)=\sqrt c\,\phi(x).
$$

### Addition

$$
k(x,z)=k_1(x,z)+k_2(x,z).
$$

The corresponding feature map is a direct sum:

$$
\phi(x)=\phi_1(x)\oplus\phi_2(x).
$$

### Multiplication

$$
k(x,z)=k_1(x,z)k_2(x,z).
$$

The classroom proof on p. 7 can be completed by using

$$
\phi(x)=\phi_1(x)\otimes\phi_2(x),
$$

because

$$
\begin{aligned}
&\langle
\phi_1(x)\otimes\phi_2(x),
\phi_1(z)\otimes\phi_2(z)
\rangle\\
&\qquad=
\langle\phi_1(x),\phi_1(z)\rangle
\langle\phi_2(x),\phi_2(z)\rangle\\
&\qquad=k_1(x,z)k_2(x,z).
\end{aligned}
$$

From the Gram-matrix perspective, the product kernel has matrix

$$
K_1\circ K_2,
$$

the elementwise product. The Schur product theorem states that the elementwise product of two positive semidefinite matrices remains positive semidefinite.

### Exponential-kernel exercise

If $k$ is a kernel, then

$$
e^{k(x,z)}
=
\sum_{m=0}^{\infty}
\frac{k(x,z)^m}{m!}
$$

is also a kernel, because:

- every $k^m$ is a product kernel;
- every coefficient $1/m!$ is nonnegative;
- a nonnegative weighted sum of kernels remains a kernel.

### Inhomogeneous polynomial kernel

$$
\boxed{
k(x,z)=
(c+\langle x,z\rangle)^d,
\qquad c\ge0,\quad d\in\mathbb N.
}
$$

The sum of a constant kernel and a linear kernel is a kernel; repeated product closure then proves the claim.

## Section 17 — Gram Matrices and General Input Spaces (L6 pp. 8–9)

Kernel methods do not require inputs to be Euclidean vectors. Inputs may be:

- strings;
- sets;
- graphs;
- group elements;
- probability distributions;
- other structured objects.

As long as a valid positive semidefinite kernel can be defined on the objects, kernel learning methods can be applied.

For examples $x_1,\ldots,x_m$,

$$
K_{ij}=k(x_i,x_j).
$$

The definition of a valid kernel can be stated entirely through its Gram matrices:

> For every sample size and every finite input set, the Gram matrix generated by $k$ must be positive semidefinite.

The closure properties can be proved again in matrix form:

- linear kernel:

  $$
  K=XX^\top\succeq0;
  $$

- nonnegative scaling:

  $$
  K\succeq0\Rightarrow cK\succeq0;
  $$

- addition:

  $$
  K_1,K_2\succeq0
  \Rightarrow K_1+K_2\succeq0;
  $$

- multiplication:

  $$
  K_1\circ K_2\succeq0.
  $$

Kernel methods replace an explicit feature matrix with an $N\times N$ Gram matrix. This avoids dependence on the feature dimension, but it is not free: storing the matrix usually costs $O(N^2)$, and solving a dense linear system directly can cost $O(N^3)$.

## Section 18 — Translation-Invariant Kernels (L6 p. 10)

A translation-invariant kernel has the form

$$
K(x,z)=\psi(x-z).
$$

In other words, translating both inputs by the same amount does not change their similarity:

$$
K(x+a,z+a)=K(x,z).
$$

### Cosine kernel

In one dimension,

$$
K(x,z)=\cos(x-z).
$$

Using

$$
\cos(x-z)=\cos x\cos z+\sin x\sin z,
$$

we may take

$$
\phi(x)=(\cos x,\sin x).
$$

Thus the cosine of the difference is a valid kernel.

### Gaussian RBF kernel

$$
\boxed{
K(x,z)=e^{-\gamma\|x-z\|^2},
\qquad \gamma>0.
}
$$

This kernel depends only on the distance between the points. Nearby points have values close to $1$, whereas far-apart points have values close to $0$.

## Section 19 — Bochner's Theorem (L6 p. 11)

Bochner's theorem completely characterizes continuous translation-invariant kernels.

Let

$$
K(x,z)=\psi(x-z),
\qquad x,z\in\mathbb R^n.
$$

Then $K$ is a positive-definite kernel if and only if there exists a finite nonnegative Borel measure $\mu$ such that

$$
\boxed{
\psi(t)
=
\int_{\mathbb R^n}
e^{i\langle t,\omega\rangle}
\,d\mu(\omega).
}
$$

Equivalently:

> Continuous translation-invariant positive-definite kernels are exactly the Fourier transforms of nonnegative measures.

The easy direction of the proof is

$$
\begin{aligned}
\sum_{i,j}\overline c_i c_jK(x_i,x_j)
&=
\int
\sum_{i,j}\overline c_i c_j
e^{i\langle x_j-x_i,\omega\rangle}
d\mu(\omega)\\
&=
\int
\left|
\sum_jc_je^{i\langle x_j,\omega\rangle}
\right|^2
d\mu(\omega)\\
&\ge0.
\end{aligned}
$$

Thus, a nonnegative spectral measure produces a positive semidefinite kernel.

The Fourier transform of a Gaussian RBF is another nonnegative Gaussian, so the RBF is a valid kernel.

It can also be proved algebraically:

$$
e^{-\gamma\|x-z\|^2}
=
e^{-\gamma\|x\|^2}
e^{-\gamma\|z\|^2}
e^{2\gamma x^\top z},
$$

after which one uses the closure properties for exponential and product kernels.

### Supplement: Mercer spectral decomposition

Under suitable compactness and integrability assumptions,

$$
k(x,z)=\sum_{m=1}^{\infty}\mu_m e_m(x)e_m(z),
\qquad
\mu_m\ge0.
$$

If

$$
f=\sum_m a_m e_m,
$$

then

$$
\boxed{
\|f\|_{\mathcal H}^2
=
\sum_m\frac{a_m^2}{\mu_m}.
}
$$

Consequently:

- a large $\mu_m$ makes the corresponding function direction inexpensive;
- a small $\mu_m$ strongly regularizes that direction.

A kernel therefore defines not only similarities between examples but also a prior preference among function-space directions.

### Supplement: the frequency-domain RKHS norm

If

$$
k(x,z)=q(x-z),
$$

Bochner's theorem requires the Fourier spectrum of $q$ to be nonnegative. The corresponding RKHS norm is

$$
\boxed{
\|f\|_{\mathcal H}^2
=
\frac1{(2\pi)^d}
\int_{\mathbb R^d}
\frac{|\widehat f(\omega)|^2}
{\widehat q(\omega)}
\,d\omega.
}
$$

The smaller $\widehat q(\omega)$ is, the larger the penalty on that frequency.

This gives a clear smoothness interpretation of common kernels:

- the Laplacian or exponential kernel corresponds to finite-order Sobolev smoothness;
- the parameters of a Matérn kernel directly control the Sobolev smoothness order;
- a Gaussian kernel has an extremely rapidly decaying spectrum, strongly penalizes high frequencies, and therefore has a smaller RKHS of smoother functions.

A “smoother kernel” normally means a smaller allowed RKHS and a stronger prior, not a larger function space.

## Section 20 — Other Kernel Functions (L6 p. 12)

The slides first ask whether

$$
\frac{1}{1+\|x-z\|}
$$

and

$$
e^{-\gamma\|x-z\|}
$$

are kernels.

In Euclidean space, the second is the positive-definite Laplacian kernel. The first admits the representation

$$
\frac1{1+r}
=
\int_0^\infty e^{-t}e^{-tr}\,dt.
$$

Therefore,

$$
\frac{1}{1+\|x-z\|}
=
\int_0^\infty
e^{-t}e^{-t\|x-z\|}\,dt,
$$

which is a nonnegative mixture of Laplacian kernels and is therefore also a kernel.

### Probability kernel

On a probability space, define for events $A$ and $B$

$$
\boxed{
k(A,B)
=
P(A\cap B)-P(A)P(B).
}
$$

This is the covariance between two indicator variables. Define

$$
\phi(A)(\omega)
=
\mathbf1_A(\omega)-P(A).
$$

Then

$$
\langle\phi(A),\phi(B)\rangle_{L^2(P)}
=
P(A\cap B)-P(A)P(B).
$$

### Conditional-expectation kernel

$$
k(x,z)
=
\mathbb E_c[p(x\mid c)p(z\mid c)].
$$

Viewing

$$
\phi(x)(c)=p(x\mid c)
$$

as an $L^2$ function of the latent variable $c$ gives the required inner-product representation.

### B-spline kernel

The slides list

$$
B_{2n+1}(x-z).
$$

Its validity can be verified through the nonnegativity of the Fourier spectrum of this translation-invariant function.

### Jaccard kernel

For nonempty sets $A$ and $B$,

$$
\boxed{
k(A,B)=\frac{|A\cap B|}{|A\cup B|}.
}
$$

This measures intersection over union and is widely used for set similarity.

The slides also mention:

- string kernels;
- graph kernels;
- kernels on groups;
- invariant kernels;
- kernels on other structured objects.

This is an important advantage of kernel methods: once an appropriate positive-definite similarity has been defined, an object need not first be crudely encoded as a fixed-length vector.

### Correction to an example in *MLbookSol*

For the shopping-basket classifier on p. 43 of *MLbookSol*, if prediction is written as

$$
\operatorname{sign}(2\langle w,\phi(x)\rangle-1),
$$

the augmented parameter should be

$$
(2w,-1),
$$

not $(2w,1)$.

For the kernel expansion of the nearest-centroid classifier on p. 44, the mean kernel value for the negative class must be subtracted:

$$
\frac1{m_+}\sum_{y_i=1}K(x,x_i)
-
\frac1{m_-}\sum_{y_i=-1}K(x,x_i).
$$

## Section 21 — Verifying That a Function Is a Kernel Can Be Very Hard (L6 p. 13)

The slides give two advanced examples. Their purpose is not to demand an immediate proof, but to illustrate that positive definiteness can involve deep mathematics.

### A BMV-related example

If $A$ is Hermitian and $B\succeq0$, then

$$
K(x,y)
=
\operatorname{tr}\exp(A-(x+y)B)
$$

is a kernel on $\mathbb R_+\times\mathbb R_+$.

This is connected to the BMV conjecture, studied from 1979 to 2011. The intuition is that if

$$
f(t)=\operatorname{tr}e^{A-tB}
$$

can be written as the Laplace transform of a nonnegative measure,

$$
f(t)=\int e^{-\lambda t}\,d\mu(\lambda),
$$

then

$$
f(x+y)
=
\int
e^{-\lambda x}e^{-\lambda y}
\,d\mu(\lambda),
$$

which directly exhibits an inner-product representation.

### A differential-entropy example

Define

$$
h(t)=\operatorname{Ent}(X+\sqrt t\,Z),
\qquad Z\sim N(0,1),
$$

where differential entropy is

$$
\operatorname{Ent}(U)
=
-\int f(u)\log f(u)\,du.
$$

The slides ask whether

$$
K(t,s)=h'(t+s)
$$

is a kernel on $\mathbb R_+\times\mathbb R_+$ and mark the question as open.

The main message of this page is that positive definiteness cannot always be verified by an elementary feature expansion; it can connect to deep questions in matrix analysis, Fourier and Laplace transforms, and information theory.

## Section 22 — Deriving Kernelized Regression (L6 pp. 14–16)

### L6 Part II: Kernel Ridge Regression

Page 14, “Back to using Kernels,” is a transition back from kernel theory to learning algorithms.

Consider ridge regression on nonlinear features:

$$
\boxed{
\min_w
\frac1N\sum_{i=1}^N
\bigl(y_i-w^\top\phi(x_i)\bigr)^2
+
\lambda\|w\|^2.
}
$$

Differentiate with respect to $w$:

$$
-\frac2N
\sum_i
\bigl(y_i-w^\top\phi(x_i)\bigr)\phi(x_i)
+
2\lambda w=0.
$$

Therefore,

$$
\lambda w
=
\frac1N
\sum_i
\bigl(y_i-w^\top\phi(x_i)\bigr)\phi(x_i).
$$

Define

$$
N\lambda\alpha_i
=
y_i-w^\top\phi(x_i).
$$

It follows that

$$
\boxed{
w=\sum_i\alpha_i\phi(x_i).
}
$$

Just as in an SVM, the optimum lies in the span of the training feature vectors.

For a new input $x$,

$$
\begin{aligned}
\hat f(x)
&=w^\top\phi(x)\\
&=\sum_i\alpha_i
\langle\phi(x_i),\phi(x)\rangle\\
&=\sum_i\alpha_i k(x_i,x).
\end{aligned}
$$

### How is $\alpha$ obtained?

Combining

$$
N\lambda\alpha_i
=
y_i-w^\top\phi(x_i)
$$

with

$$
w=\sum_j\alpha_j\phi(x_j)
$$

gives

$$
N\lambda\alpha_i
=
y_i-\sum_j\alpha_jk(x_j,x_i).
$$

Define

$$
K_{ij}=k(x_i,x_j).
$$

Then

$$
N\lambda\alpha=y-K\alpha.
$$

Thus

$$
(K+N\lambda I)\alpha=y,
$$

so

$$
\boxed{
\alpha=(K+N\lambda I)^{-1}y.
}
$$

The final predictor is

$$
\boxed{
\hat f(x)
=
\sum_{i=1}^N\alpha_i k(x_i,x)
=
k_x^\top(K+N\lambda I)^{-1}y,
}
$$

where

$$
k_x=(k(x_1,x),\ldots,k(x_N,x))^\top.
$$

### Coefficient forms in KRR and kernel SVM

- SVM:

  $$
  w=\sum_i\alpha_i y_i\phi(x_i);
  $$

- KRR:

  $$
  w=\sum_i\alpha_i\phi(x_i).
  $$

In KRR, the label information is already contained in

$$
\alpha=(K+N\lambda I)^{-1}y.
$$

### Supplement: representation and uniqueness

Source: *ltfp*, Sections 7.4 and 7.6, PDF pp. 212–236; *MLbookSol*, p. 42.

For

$$
\min_{f\in\mathcal H}
\frac1n\sum_i(y_i-f(x_i))^2
+\lambda\|f\|_{\mathcal H}^2,
$$

the solution has the form

$$
f(x)=\sum_i\alpha_i k(x,x_i),
$$

and a canonical coefficient solution is

$$
\boxed{
\alpha=(K+n\lambda I)^{-1}y.
}
$$

If $K$ is singular, the coefficient representation can be nonunique: adding $v\in\ker K$ does not change the represented function. The regularized prediction function itself is nevertheless unique.

### Supplement: kernel approximations

Explicitly forming and factoring $K\in\mathbb R^{n\times n}$ normally requires $O(n^2)$ memory and $O(n^3)$ time.

A Nyström approximation selects $m$ columns:

$$
K
\approx
K(:,I)K(I,I)^{-1}K(I,:),
$$

at a cost of approximately $O(nm^2)$.

Random features use a representation

$$
k(x,z)
=
\mathbb E_v[\phi(x,v)\phi(z,v)]
$$

to construct

$$
\tilde k(x,z)
=
\frac1m\sum_{j=1}^m
\phi(x,v_j)\phi(z,v_j).
$$

The distinction is:

- Nyström approximation depends on the observed training inputs;
- random features can be sampled before the data are observed.

### Correction to the KRR calculation in *MLbookSol*

If the objective on p. 42 is written as

$$
\lambda\alpha^\top G\alpha
+\frac1{2m}\|G\alpha-y\|^2,
$$

exact differentiation produces $G+2m\lambda I$, not $G+m\lambda I$. The discrepancy depends on whether the regularizer includes a factor $1/2$. The mathematical structure is unchanged, but the scaling convention for $\lambda$ must be kept consistent.

## Section 23 — Bias–Variance Decomposition for Kernel Ridge Regression (L6 p. 17)

Let the training-output vector be

$$
y=z+\varepsilon,
\qquad
z=\mathbb E[y],
\qquad
\varepsilon\sim\mathcal N(0,C).
$$

The KRR fitted values at the training points are

$$
\hat z
=
K(K+n\lambda I)^{-1}y.
$$

Define the smoothing matrix

$$
S_\lambda=K(K+n\lambda I)^{-1}.
$$

Then

$$
\hat z=S_\lambda y.
$$

The predictive error decomposes as

$$
\frac1n
\mathbb E_\varepsilon\|\hat z-z\|^2
=
\frac1n\|\mathbb E\hat z-z\|^2
+
\frac1n\operatorname{tr}\operatorname{Var}(\hat z).
$$

Because

$$
I-K(K+n\lambda I)^{-1}
=
n\lambda(K+n\lambda I)^{-1},
$$

the squared-bias term is

$$
\boxed{
\operatorname{bias}(K)
=
n\lambda^2
z^\top(K+n\lambda I)^{-2}z.
}
$$

The quantity called “bias” in the slides is in fact the normalized squared bias.

The variance term is

$$
\boxed{
\operatorname{variance}(K)
=
\frac1n
\operatorname{tr}\left[
CK^2(K+n\lambda I)^{-2}
\right].
}
$$

If

$$
K=U\operatorname{diag}(\kappa_r)U^\top,
$$

then the signal shrinkage factor in kernel eigendirection $r$ is

$$
\frac{\kappa_r}{\kappa_r+n\lambda}.
$$

As $\lambda$ increases:

- the fraction of the signal retained decreases;
- the bias factor

  $$
  \left(
  \frac{n\lambda}{\kappa_r+n\lambda}
  \right)^2
  $$

  increases;

- the noise-transmission factor

  $$
  \left(
  \frac{\kappa_r}{\kappa_r+n\lambda}
  \right)^2
  $$

  decreases.

This is the bias–variance tradeoff in kernel ridge regression.

### Supplement: KRR is a linear smoother of the labels

Source: *ltfp*, Sections 7.4 and 7.6, PDF pp. 212–236; *MLbookSol*, p. 42.

Let

$$
q(x)=
(k(x,x_1),\ldots,k(x,x_n))^\top.
$$

Then

$$
\hat f(x)
=
q(x)^\top(K+n\lambda I)^{-1}y
=
\sum_iw_i(x)y_i,
$$

where

$$
w(x)=(K+n\lambda I)^{-1}q(x).
$$

Predictions at the training points are

$$
\hat y=H_\lambda y,
\qquad
H_\lambda=K(K+n\lambda I)^{-1}.
$$

Unlike Nadaraya–Watson weights:

- KRR weights can be negative;
- they generally do not sum to one;
- negative weights permit higher-order cancellation and therefore allow KRR to exploit greater smoothness of the target function.

### Supplement: population effective dimension

Under random design, define the covariance operator

$$
\Sigma=\mathbb E[\phi(X)\otimes\phi(X)].
$$

It connects the RKHS norm to prediction error:

$$
\|g\|_{L^2(p)}^2
=
\langle g,\Sigma g\rangle_{\mathcal H}.
$$

The effective dimension is

$$
\boxed{
\mathcal N(\lambda)
=
\operatorname{tr}\bigl[
\Sigma(\Sigma+\lambda I)^{-1}
\bigr].
}
$$

The main structure of the KRR risk can be written as

$$
\mathbb E\|\hat f_\lambda-f_*\|_{L^2(p)}^2
\lesssim
\frac{\sigma^2}{n}\mathcal N(\lambda)
+
A(\lambda,f_*),
$$

where

$$
A(\lambda,f_*)
=
\inf_{f\in\mathcal H}
\left\{
\|f-f_*\|_{L^2(p)}^2
+\lambda\|f\|_{\mathcal H}^2
\right\}.
$$

The first term is the estimation or variance contribution; the second is regularized approximation bias.

### Supplement: smoothness-adaptive rates for KRR

Suppose the kernel eigenvalues have Sobolev-type decay

$$
\mu_j\asymp j^{-2s/d}
$$

and the target function has smoothness order $t$. Typically,

$$
\mathcal N(\lambda)
\asymp
\lambda^{-d/(2s)},
\qquad
A(\lambda,f_*)\asymp\lambda^{t/s}.
$$

Balancing these terms yields

$$
\boxed{
\lambda_{\rm opt}
\asymp
n^{-2s/(2t+d)}
}
$$

and

$$
\boxed{
\mathbb E\|\hat f-f_*\|^2
\asymp
n^{-2t/(2t+d)}.
}
$$

When $t=1$, this recovers the local-averaging rate

$$
n^{-2/(d+2)}.
$$

The smoother the target, the faster the rate that KRR can attain. This is what it means for a kernel method to adapt to target smoothness.

## Section 24 — Why $\lambda\approx0$ Can Sometimes Be Best (L6 p. 18)

The slides show the MNIST binary-classification experiments of Liang and Rakhlin (2018), using a Gaussian kernel. The tested digit pairs include

$$
[2,5],\ [2,9],\ [3,6],\ [3,8],\ [4,7].
$$

In the plots, test error is smallest near $\lambda=0$ and then increases as $\lambda$ grows.

At first this seems to contradict the claim that regularization is needed to prevent overfitting. The crucial point is that

$$
\lambda=0
$$

does not necessarily mean that the model has no preference at all.

When the kernel matrix can interpolate the training labels, an algorithm often selects:

- the minimum-RKHS-norm interpolating solution;
- or the minimum-norm solution implicitly selected by a pseudoinverse or numerical solver.

Thus, strong implicit regularization can be present even without an explicit ridge penalty.

When the dimension, kernel spectrum, and data structure are favorable, interpolating noise need not cause large test error. This phenomenon is called *benign overfitting* or *ridgeless regression*.

The figure does not say that regularization is always useless. Rather:

> The best $\lambda$ depends on the data, kernel, noise, and algorithm; the classical U-shaped curve is not universal in every high-dimensional problem.

### Supplement: zero-initialized gradient descent selects the minimum-norm interpolator

Source: *ltfp*, Sections 12.1–12.2, PDF pp. 359–375.

When $d>n$ and $XX^\top$ is invertible, the equation

$$
X\theta=y
$$

has infinitely many solutions. Starting gradient descent from

$$
\theta_0=0
$$

keeps every iterate in the row space of $X$, and the iterates converge to

$$
\boxed{
\theta^\dagger
=
X^\top(XX^\top)^{-1}y.
}
$$

This is the Euclidean minimum-norm solution among all interpolators. The kernel analogue is the minimum-RKHS-norm interpolator.

Therefore, the claim that “$\lambda=0$ can still generalize” depends on which interpolator the algorithm selects. Not every zero-training-error solution generalizes well.

### Supplement: an exact double-descent example

Suppose

$$
x\sim\mathcal N(0,I_d),
\qquad
y=x^\top\theta_*+\epsilon,
\qquad
\operatorname{Var}(\epsilon)=\sigma^2.
$$

For the minimum-norm empirical risk minimizer, when $d\le n-2$,

$$
\mathbb E[R(\hat\theta)-R^*]
=
\frac{\sigma^2d}{n-d-1}.
$$

When $d\ge n+2$,

$$
\mathbb E[R(\hat\theta)-R^*]
=
\frac{\sigma^2n}{d-n-1}
+
\|\theta_*\|^2\frac{d-n}{d}.
$$

Near the interpolation threshold $d\approx n$, inverse-matrix variance blows up. After crossing the threshold, variance falls again, but projection bias appears, producing the second descent.

If ridge's $\lambda$ is retuned separately for each $d$, the double-descent peak is usually greatly reduced and can even disappear.

## Section 25 — What Is an RKHS? (L6 p. 19)

### L6 Part III: RKHSs and the Representer Theorem

RKHS stands for *reproducing kernel Hilbert space*.

Let $\mathcal H$ be a Hilbert space of functions

$$
f:\mathcal X\to\mathbb R.
$$

If there is a kernel $k$ such that, for every $f\in\mathcal H$,

$$
\boxed{
\langle f,k(\cdot,x)\rangle_{\mathcal H}
=
f(x),
}
$$

then $\mathcal H$ is an RKHS.

This is the reproducing property: the value of a function at $x$ is recovered by taking its inner product with $k(\cdot,x)$.

Letting $f=k(\cdot,x')$ gives

$$
\boxed{
\langle k(\cdot,x'),k(\cdot,x)\rangle_{\mathcal H}
=
k(x',x).
}
$$

A natural feature map is therefore

$$
\phi(x)=k(\cdot,x).
$$

Moreover,

$$
\mathcal H
=
\overline{
\operatorname{span}
\{k(\cdot,x):x\in\mathcal X\}
}.
$$

In other words, the functions in an RKHS can be constructed from finite linear combinations of kernel sections $k(\cdot,x)$ and limits of such combinations.

### Supplement: constructing an RKHS directly from a kernel

Source: *ltfp*, Sections 7.2–7.3, PDF pp. 197–201.

First define

$$
\mathcal H_0
=
\left\{
\sum_i\alpha_i k(\cdot,x_i)
\right\}
$$

with inner product

$$
\left\langle
\sum_i\alpha_i k(\cdot,x_i),
\sum_j\beta_j k(\cdot,z_j)
\right\rangle
=
\sum_{i,j}\alpha_i\beta_jk(x_i,z_j).
$$

Completing this space yields an RKHS, and it automatically satisfies the reproducing property

$$
\boxed{
\langle k(\cdot,x),f\rangle_{\mathcal H}
=f(x).
}
$$

By Cauchy–Schwarz,

$$
\boxed{
|f(x)|
\le
\|f\|_{\mathcal H}\sqrt{k(x,x)}.
}
$$

Thus, controlling the RKHS norm also controls pointwise function values.

An ordinary $L^2$ space is not an RKHS. Its functions are defined only up to equality almost everywhere, so evaluation at a single point is not a continuous functional.

## Section 26 — The Representer Theorem (L6 p. 20)

Consider a general regularized empirical-risk problem:

$$
\min_{f\in\mathcal H}
R_n\bigl(
f(x^{(1)}),\ldots,f(x^{(n)})
\bigr)
+
\lambda\Omega(f).
$$

Suppose the regularizer is nondecreasing in the Hilbert norm, meaning

$$
\|f\|_{\mathcal H}>\|g\|_{\mathcal H}
\quad\Longrightarrow\quad
\Omega(f)\ge\Omega(g).
$$

Then at least one optimum has a finite expansion:

$$
\boxed{
f(\cdot)
=
\sum_{i=1}^n
\alpha_i k(\cdot,x^{(i)}).
}
$$

If $\Omega$ is strictly increasing in the norm, every optimum has this form.

### Why does the theorem hold?

Let

$$
S=
\operatorname{span}
\{k(\cdot,x^{(1)}),\ldots,k(\cdot,x^{(n)})\}.
$$

Every $f\in\mathcal H$ has the orthogonal decomposition

$$
f=f_\parallel+f_\perp,
\qquad
f_\parallel\in S,
\qquad
f_\perp\perp S.
$$

At every training point,

$$
\begin{aligned}
f(x^{(i)})
&=
\langle f,k(\cdot,x^{(i)})\rangle\\
&=
\langle f_\parallel,k(\cdot,x^{(i)})\rangle,
\end{aligned}
$$

because

$$
\langle f_\perp,k(\cdot,x^{(i)})\rangle=0.
$$

Deleting $f_\perp$ therefore changes neither the predictions at the training points nor the loss. On the other hand,

$$
\|f\|^2
=
\|f_\parallel\|^2+\|f_\perp\|^2
\ge
\|f_\parallel\|^2.
$$

If the regularizer is nondecreasing in the norm, deleting $f_\perp$ cannot worsen the objective. An optimum can therefore be chosen entirely inside the finite-dimensional subspace $S$.

This explains why an optimization problem that may originally be infinite-dimensional ultimately needs only $n$ coefficients.

Two equivalent expressions are:

- feature-space vector:

  $$
  w=\sum_i\alpha_i\phi(x_i);
  $$

- RKHS function:

  $$
  f(\cdot)=\sum_i\alpha_i k(x_i,\cdot).
  $$

Page 21 recommends the sections of the course text concerning linear prediction, logistic regression, SVMs, and kernels, together with the SVM/kernel notes on Moodle.

### Supplement: a more general form of the representer theorem

Consider an objective

$$
\Psi\bigl(
f(x_1),\ldots,f(x_n),\|f\|_{\mathcal H}^2
\bigr).
$$

If $\Psi$ is strictly increasing in its final norm argument, every optimum can be written as

$$
\boxed{
f=\sum_{i=1}^n\alpha_i k(\cdot,x_i).
}
$$

The proof decomposes

$$
f=f_D+f_\perp
$$

into a component in the span of the training features and an orthogonal component. The orthogonal component does not change any training prediction but increases the norm, so it must vanish at an optimum.

Crucially, the representer theorem itself does not require the loss to be convex. Convexity is mainly used to make optimization tractable, to establish uniqueness, and to carry out dual analysis.

### Supplement: minimum-norm interpolation

If an RKHS function satisfying

$$
f(x_i)=y_i
$$

exists, then the minimum-norm interpolator satisfies

$$
f(\cdot)
=
\sum_i\alpha_i k(\cdot,x_i),
\qquad
K\alpha=y.
$$

This is the basis of ridgeless KRR and minimum-norm implicit bias.

## Section 27 — Kernel Density Estimation (L6 p. 23)

### L6 Part IV: KDE, Kernel Regression, and Attention

Page 22, “Kernels and Attention,” is the transition into this final part of the lecture.

Given one-dimensional iid samples

$$
x_1,\ldots,x_n\sim f,
$$

the goal is to estimate the unknown density $f(x)$.

The kernel density estimator is

$$
\boxed{
\hat f_h(x)
=
\frac1n\sum_{i=1}^nK_h(x-x_i)
=
\frac1{nh}
\sum_{i=1}^n
K\left(\frac{x-x_i}{h}\right).
}
$$

Here:

- $K$ is a smoothing window;
- $h>0$ is the bandwidth.

A standard KDE normally requires

$$
\int K(u)\,du=1,
$$

and commonly uses $K(u)\ge0$. The slides also impose the zero-first-moment condition

$$
\int uK(u)\,du=0,
$$

which helps eliminate the first-order bias term.

### The bandwidth bias–variance tradeoff

- Very small $h$: every example produces a narrow peak. The estimate tracks local details but fluctuates sharply, giving low bias and high variance.
- Very large $h$: the curve is smooth and stable but can erase genuine structure, giving high bias and low variance.

In the slide figure:

- $h=0.3$: relatively smooth;
- $h=0.1$: more local structure is retained;
- $h=0.05$: the estimate is visibly too variable.

In practice, bandwidth can be selected with cross-validation or a plug-in method. In one dimension, when the density is sufficiently smooth, the typical asymptotic scaling is

$$
h_{\mathrm{opt}}\propto n^{-1/5}.
$$

### Warning: “kernel” has different meanings

In KDE, $K$ is primarily a smoothing window. It need not be a positive semidefinite Mercer kernel in the earlier RKHS sense.

## Section 28 — Nadaraya–Watson Kernel Regression (L6 p. 24)

The regression target is the conditional mean

$$
m(x)=\mathbb E[Y\mid X=x].
$$

Using conditional density,

$$
\mathbb E[Y\mid X=x]
=
\int y f(y\mid x)\,dy
=
\int y\frac{f(x,y)}{f(x)}\,dy.
$$

Estimate the joint density with a product kernel:

$$
\hat f(x,y)
=
\frac1n
\sum_{i=1}^n
K_h(x-x_i)K_h(y-y_i),
$$

and estimate the marginal density by

$$
\hat f(x)
=
\frac1n
\sum_{i=1}^n
K_h(x-x_i).
$$

Substituting these estimates into the conditional mean and using a normalized, zero-mean kernel for which

$$
\int yK_h(y-y_i)\,dy=y_i
$$

gives

$$
\boxed{
\hat m(x)
=
\frac{
\sum_iK_h(x-x_i)y_i
}{
\sum_jK_h(x-x_j)
}.
}
$$

Define

$$
w_i(x)
=
\frac{K_h(x-x_i)}
{\sum_jK_h(x-x_j)}.
$$

Then

$$
\boxed{
\hat m(x)=\sum_iw_i(x)y_i.
}
$$

If $K\ge0$, the weights are nonnegative and sum to one. The estimator is therefore a weighted nearest-neighbor method:

- examples closer to the query receive larger weights;
- the prediction is a weighted average of nearby labels.

### Supplement: the Nadaraya–Watson kernel need not be PSD

Source: *ltfp*, Sections 6.2–6.5, PDF pp. 171–194.

Nadaraya–Watson weights have the form

$$
w_i(x)
=
\frac{k(x,x_i)}
{\sum_jk(x,x_j)},
\qquad
w_i(x)\ge0,
\qquad
\sum_iw_i(x)=1.
$$

Here, the kernel is required only to be pointwise nonnegative so that it can construct local weights. It need not generate a positive semidefinite Gram matrix.

This is a different concept from the Mercer kernels used in SVMs and KRR. A Gaussian happens to satisfy both definitions; that coincidence does not make the two definitions identical.

### Supplement: conditional bias–variance decomposition for NW

If

$$
\hat f(x)=\sum_iw_i(x)y_i,
\qquad
f_*(x)=\mathbb E[Y\mid X=x],
$$

then, conditional on the training inputs,

$$
\begin{aligned}
&\mathbb E\left[
(\hat f(x)-f_*(x))^2\mid X_{1:n}
\right]\\
&\quad=
\left[
\sum_iw_i(x)(f_*(x_i)-f_*(x))
\right]^2
+
\sum_iw_i(x)^2
\operatorname{Var}(Y_i\mid x_i).
\end{aligned}
$$

If $f_*$ is $B$-Lipschitz and the noise variance is at most $\sigma^2$, this is bounded by

$$
B^2\sum_iw_i(x)\|x_i-x\|^2
+
\sigma^2\sum_iw_i(x)^2.
$$

More concentrated weights usually lower bias, but they increase

$$
\sum_iw_i^2
$$

and therefore raise variance.

### Supplement: bandwidth and the curse of dimensionality

For a $d$-dimensional Lipschitz regression function,

$$
\operatorname{Variance}
\asymp
\frac{\sigma^2}{nh^d},
\qquad
\operatorname{Bias}^2
\asymp
B^2h^2.
$$

Consequently,

$$
\boxed{
h_{\rm opt}\asymp n^{-1/(d+2)}
}
$$

and

$$
\boxed{
\operatorname{Risk}\asymp n^{-2/(d+2)}.
}
$$

Consistency requires both

$$
h\to0,
\qquad
nh^d\to\infty.
$$

If the target is twice differentiable and the kernel has suitable symmetry, squared bias improves to $O(h^4)$, leading to

$$
h_{\rm opt}\asymp n^{-1/(d+4)},
\qquad
\operatorname{Risk}\asymp n^{-4/(d+4)}.
$$

In particular:

- for a one-dimensional, twice-smooth KDE or NW problem, one commonly sees $h\asymp n^{-1/5}$;
- for one-dimensional NW regression under only a Lipschitz assumption, $h\asymp n^{-1/3}$.

The two conclusions arise from different smoothness assumptions and must not be mixed.

## Section 29 — From Nadaraya–Watson Regression to Attention

Attention can be written as

$$
\boxed{
\operatorname{Attn}(q;K,V)
=
\frac{
\sum_j\kappa_\theta(q,k_j)v_j
}{
\sum_\ell\kappa_\theta(q,k_\ell)
}.
}
$$

Scaled dot-product attention uses

$$
\kappa_\theta(q,k_j)
=
\exp\left(
\frac{q^\top k_j}{\sqrt d}
\right).
$$

The correspondence is:

| Kernel regression | Attention |
|---|---|
| query input $x$ | query $q$ |
| training input $x_j$ | key $k_j$ |
| label $y_j$ | value $v_j$ |
| $K_h(x-x_j)$ | $\kappa_\theta(q,k_j)$ |
| normalized kernel weight | softmax weight |
| weighted average of labels | weighted average of values |

Both have the common form

$$
\text{output}
=
\frac{
\sum_j
(\text{similarity})_j
(\text{value})_j
}{
\sum_j(\text{similarity})_j
}.
$$

### But they are not identical

- Kernel-regression similarity is usually determined by a fixed distance and a hand-selected bandwidth.
- Attention learns its query and key representations, and therefore its similarity, from data.
- The value in kernel regression is usually a scalar label, whereas an attention value is normally a vector.
- Attention may use different projections for queries and keys, so its similarity need not be symmetric.
- Attention can include masks, multiple heads, positional encodings, and causal constraints.
- Kernel regression normally estimates a conditional mean from iid samples; attention is a context-aggregation mechanism inside a neural network.
- Attention and kernel regression have similar algebraic forms, but attention need not be a Mercer-kernel method.

### Supplement: NW, KRR, and attention compared

Source for attention discussion: *ltfp*, PDF pp. 294–295.

| Method | Similarity requirement | Weights | Main control quantity |
|---|---|---|---|
| NW | the smoothing window is pointwise nonnegative | nonnegative and sum to one | bandwidth $h$ |
| KRR | the Gram matrix is PSD | may be negative and generally do not sum to one | regularization $\lambda$ |
| Attention | softmax scores | nonnegative and each row sums to one | learned $Q,K,V$ |

The standard form of attention is

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(
\frac{QK^\top}{\sqrt d}
\right)V.
$$

It resembles a normalized weighted average of values and can therefore be compared structurally with NW. This is only a structural analogy:

- $Q$, $K$, and $V$ depend jointly on the data and learned parameters;
- the query and key maps may differ;
- $QK^\top$ need not be symmetric;
- it need not be a PSD Gram matrix.

Consequently, the claim that “attention is NW/KRR” is not a theorem established by the two supplementary books.

### Final synthesis of L5 and L6

The central logic of the two lectures can be compressed into nine steps without discarding any of the details developed above:

1. The OLS solution depends on $(X^\top X)^{-1}$, which may not exist or may be unstable.
2. Ridge regression uses $\lambda I$ to stabilize the solution and shrink high-variance directions.
3. Regularization introduces bias but can improve test error by reducing variance by a larger amount.
4. When a linear model is not expressive enough, one can first construct nonlinear features $\phi(x)$.
5. Explicit features may grow combinatorially or even form an infinite-dimensional space.
6. The optima of SVMs, KRR, and related methods lie in the span of the training-example features.
7. Hence the algorithms require only inner products

   $$
   \langle\phi(x_i),\phi(x_j)\rangle.
   $$

8. Replacing these inner products with a valid positive semidefinite kernel

   $$
   k(x_i,x_j)
   =
   \langle\phi(x_i),\phi(x_j)\rangle
   $$

   gives a kernel method.
9. The representer theorem explains, in the general theory of RKHSs, why these finite kernel expansions occur so broadly.

Always distinguish two independent control knobs:

- the kernel $k$ selects the feature space, the meaning of similarity, and what the model can express;
- the regularization parameter $\lambda$ controls shrinkage and the bias–variance tradeoff inside the chosen feature space.

Also distinguish three uses of the word “kernel”:

- in SVMs, KRR, and RKHSs, it is an inner-product kernel that must generate positive semidefinite Gram matrices;
- in KDE and Nadaraya–Watson regression, it is a local smoothing window and need not be a Mercer kernel;
- in attention, it is a learned similarity weight with a form reminiscent of kernel regression but not identical to a traditional kernel method.

The two book supplements and the original L5/L6 notes are now integrated while preserving the assumptions and source boundaries of every formula.